# Windowed SBS Analysis

## Config

In [18]:
# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
import os, re, shutil, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.optimize import nnls
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform, cosine
from concurrent.futures import ProcessPoolExecutor, as_completed
from SigProfilerMatrixGenerator.scripts import SigProfilerMatrixGeneratorFunc as matGen

MUTATION_DIR  = "/home/alexpalazzo1/Documents/Tina/COSMIC/mutations_1kb/"
BED_1KB_DIR   = "/home/alexpalazzo1/Documents/Tina/COSMIC/bedfiles_1kb/"
COSMIC_REF    = "/home/alexpalazzo1/Documents/Tina/COSMIC/COSMIC_v3.5_SBS_GRCh38.txt"
GENOME_FASTA  = "/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa"

# Output dirs — one per analysis version
OUT_NORM    = "/home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_nothresh/"
OUT_NONORM  = "/home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_no_norm_nothresh/"
OUT_SHARED  = "/home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_nothresh/"

TMP_NORM    = "/home/alexpalazzo1/Documents/Tina/COSMIC/tmp_windowed_no_somatic_dnm_nothresh/"
TMP_NONORM  = "/home/alexpalazzo1/Documents/Tina/COSMIC/tmp_windowed_no_somatic_dnm_no_norm_nothresh/"

for d in [OUT_NORM, OUT_NONORM, OUT_SHARED, TMP_NORM, TMP_NONORM]:
    os.makedirs(d, exist_ok=True)

WINDOW_SIZE        = 50
N_WINDOWS          = 20
ACTIVITY_THRESHOLD = 0.05

N_WORKERS          = max(1, os.cpu_count() - 2)

REGION_FILES = {
    "All_protein_coding_genes":      (os.path.join(MUTATION_DIR, "Protein_coding_gene_rareSNP.txt"), True, 14, True),
    "lncRNA_expressed_in_testes":                      (os.path.join(MUTATION_DIR, "lncRNA.txt"),                      True, 14, True),
    "Protein_coding_genes_not_expressed_in_testes":   (os.path.join(MUTATION_DIR, "Non_testes_expressed_gene.txt"),   True, 14, True),
    "lncRNA_not_expressed_in_testes": (os.path.join(MUTATION_DIR, "Non_testes_expressed_lncRNA.txt"), True, 14, True),
    "Intergenic_RNAPII_pause_sites":                      (os.path.join(MUTATION_DIR, "RNAPII.txt"),                      True, 14, True),
    "Random_intergenic":           (os.path.join(MUTATION_DIR, "Intergenic_random.txt"),           True, 15, True),
}

PLOT_ORDER = [
    "All_protein_coding_genes",
    "lncRNA_expressed_in_testes",
    "Intergenic_RNAPII_pause_sites",
    "Protein_coding_genes_not_expressed_in_testes",
    "lncRNA_not_expressed_in_testes",
    "Random_intergenic",
]

UNIVERSAL_ORDER = [
    "SBS98", "SBS15", "SBS6", "SBS1", "SBS87", "SBS24", "SBS31",
    "SBS19", "SBS23", "SBS11", "SBS32", "SBS7b", "SBS102", "SBS30",
    "SBS39", "SBS86", "SBS57", "SBS16", "SBS3", "SBS40c", "SBS5",
    "SBS54", "SBS46", "SBS12", "SBS26"
]
UNIVERSAL_ORDER = ['SBS48', 'SBS91', 'SBS59', 'SBS106', 'SBS43', 'SBS55', 'SBS60', 'SBS10b', 'SBS2', 'SBS17b', 'SBS28', 'SBS57', 'SBS17a', 'SBS33', 'SBS54', 'SBS27', 'SBS98', 'SBS1', 'SBS87', 'SBS13', 'SBS103', 'SBS39', 'SBS86', 'SBS49', 'SBS53', 'SBS10c', 'SBS10a', 'SBS10d', 'SBS14', 'SBS20', 'SBS35', 'SBS24', 'SBS29', 'SBS45', 'SBS8', 'SBS94', 'SBS100', 'SBS4', 'SBS107', 'SBS15', 'SBS6', 'SBS99', 'SBS42', 'SBS84', 'SBS105', 'SBS44', 'SBS97', 'SBS31', 'SBS19', 'SBS23', 'SBS7a', 'SBS7b', 'SBS102', 'SBS30', 'SBS110', 'SBS40a', 'SBS11', 'SBS32', 'SBS58', 'SBS50', 'SBS51', 'SBS7c', 'SBS34', 'SBS85', 'SBS7d', 'SBS21', 'SBS46', 'SBS37', 'SBS12', 'SBS26', 'SBS16', 'SBS88', 'SBS22a', 'SBS101', 'SBS25', 'SBS40b', 'SBS9', 'SBS96', 'SBS92', 'SBS40c', 'SBS5', 'SBS3', 'SBS89']
print(f"Regions: {list(REGION_FILES.keys())}")
print(f"Workers: {N_WORKERS}")
print(f"Windows: {N_WINDOWS} x {WINDOW_SIZE}bp")


Regions: ['All_protein_coding_genes', 'lncRNA_expressed_in_testes', 'Protein_coding_genes_not_expressed_in_testes', 'lncRNA_not_expressed_in_testes', 'Intergenic_RNAPII_pause_sites', 'Random_intergenic']
Workers: 30
Windows: 20 x 50bp


## Steps 1-3: Parse Mutations → Write VCFs → Generate Matrices
*(Skip and use Reload cell below if already computed)*

In [2]:
# ─────────────────────────────────────────────────────────────
# STEP 1: Parse mutation files → assign each mutation to a window
# Shared by both analysis versions
# ─────────────────────────────────────────────────────────────

def parse_mutation_file(name, path, has_strand, ncols, apply_rc):
    rows = []
    with open(path) as f:
        for line in f:
            cols = line.strip().split()
            if len(cols) < 4:
                continue
            try:
                rel_pos = int(cols[0])
                mut     = cols[1]
                if ">" not in mut:
                    continue
                parts = mut.split(">")
                ref = parts[0][-1]
                alt = parts[1][0]
                if ref not in "ACGT" or alt not in "ACGT":
                    continue
                strand = cols[-1] if has_strand else "+"
                pos    = int(cols[-2])
                chrom  = str(cols[-3])
                window = min(rel_pos // WINDOW_SIZE, N_WINDOWS - 1)
                rows.append({"rel_pos": rel_pos, "window": window, "ref": ref,
                              "alt": alt, "chrom": chrom, "pos": pos,
                              "strand": strand, "apply_rc": apply_rc})
            except (ValueError, IndexError):
                continue

    if len(rows) == 0:
        print(f"  WARNING: {name} — no valid mutations parsed")
        return pd.DataFrame(columns=["rel_pos","window","ref","alt","chrom","pos","strand","apply_rc"])

    df = pd.DataFrame(rows)
    print(f"  {name}: {len(df)} mutations loaded across {df['window'].nunique()} windows")
    return df

all_mutations = {}
for name, (path, has_strand, ncols, apply_rc) in REGION_FILES.items():
    all_mutations[name] = parse_mutation_file(name, path, has_strand, ncols, apply_rc)

print("\nMutation loading complete.")


  All_protein_coding_genes: 3866951 mutations loaded across 20 windows
  lncRNA_expressed_in_testes: 938001 mutations loaded across 20 windows
  Protein_coding_genes_not_expressed_in_testes: 161025 mutations loaded across 20 windows
  lncRNA_not_expressed_in_testes: 2324918 mutations loaded across 20 windows
  Intergenic_RNAPII_pause_sites: 276173 mutations loaded across 20 windows
  Random_intergenic: 2417493 mutations loaded across 20 windows

Mutation loading complete.


In [3]:
# ─────────────────────────────────────────────────────────────
# STEP 2: Write VCFs per window
# NOTE: VCFs are written to TMP_NORM — reused by both versions
# since mutations are the same; only the fitting differs
# ─────────────────────────────────────────────────────────────

COMPLEMENT = str.maketrans("ACGT", "TGCA")

def mut_to_vcf_row(chrom, pos, ref, alt, strand, apply_rc):
    if apply_rc and strand == "-":
        ref = ref.translate(COMPLEMENT)
        alt = alt.translate(COMPLEMENT)
    chrom = chrom.replace("chr", "")
    return f"{chrom}\t{pos}\t.\t{ref}\t{alt}\t.\t.\t.\n"

def write_window_vcf_task(args):
    name, w, df_window_records, vcf_dir = args
    os.makedirs(vcf_dir, exist_ok=True)
    vcf_path = os.path.join(vcf_dir, f"{name}_w{w:02d}.vcf")
    with open(vcf_path, "w") as f:
        f.write("##fileformat=VCFv4.1\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        seen = set()
        for row in df_window_records:
            key = (row["chrom"], row["pos"], row["ref"], row["alt"])
            if key in seen:
                continue
            seen.add(key)
            f.write(mut_to_vcf_row(row["chrom"], row["pos"],
                                    row["ref"], row["alt"],
                                    row["strand"], row["apply_rc"]))
    return (name, w, vcf_dir)

tasks = []
for name, df in all_mutations.items():
    for w in range(N_WINDOWS):
        df_w = df[df["window"] == w]
        if len(df_w) == 0:
            continue
        vcf_dir = os.path.join(TMP_NORM, f"{name}_w{w:02d}")
        os.makedirs(vcf_dir, exist_ok=True)
        records = df_w[["chrom","pos","ref","alt","strand","apply_rc"]].to_dict("records")
        tasks.append((name, w, records, vcf_dir))

print(f"Writing {len(tasks)} VCF files using {N_WORKERS} workers...")
window_vcf_dirs = {name: {} for name in all_mutations}

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(write_window_vcf_task, t): t for t in tasks}
    completed = 0
    for future in as_completed(futures):
        name, w, vcf_dir = future.result()
        window_vcf_dirs[name][w] = vcf_dir
        completed += 1
        if completed % 40 == 0:
            print(f"  {completed}/{len(tasks)} VCFs written...")

print(f"VCF writing complete. {completed} files written.")


Writing 120 VCF files using 30 workers...
  40/120 VCFs written...
  80/120 VCFs written...
  120/120 VCFs written...
VCF writing complete. 120 files written.


In [4]:
# ─────────────────────────────────────────────────────────────
# STEP 3: Generate SBS96 mutation matrices (shared by both versions)
# ─────────────────────────────────────────────────────────────

def generate_matrix_for_window(args):
    name, w, vcf_dir = args
    try:
        matrices = matGen.SigProfilerMatrixGeneratorFunc(
            f"{name}_w{w:02d}", "GRCh38", vcf_dir,
            exome=False, bed_file=None, chrom_based=False,
            plot=False, seqInfo=False,
        )
        if "96" in matrices:
            return (name, w, matrices["96"].sum(axis=1))
        return (name, w, None)
    except Exception:
        return (name, w, None)

tasks = [(name, w, vcf_dir)
         for name, windows in window_vcf_dirs.items()
         for w, vcf_dir in windows.items()]

print(f"Running SigProfilerMatrixGenerator for {len(tasks)} combinations using {N_WORKERS} workers...")
window_mut_matrices = {name: {} for name in REGION_FILES}

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(generate_matrix_for_window, t): t for t in tasks}
    completed = 0
    for future in as_completed(futures):
        name, w, result = future.result()
        if result is not None:
            window_mut_matrices[name][w] = result
        completed += 1
        if completed % 20 == 0:
            print(f"  {completed}/{len(tasks)} complete...")

print("\nMatrix generation complete!")
for name in REGION_FILES:
    print(f"  {name}: {len(window_mut_matrices[name])} windows with data")

# ── SAVE: matrix data so we never need to rerun Steps 1-3 ──
save_path = os.path.join(OUT_SHARED, "window_mut_matrices.pkl")
with open(save_path, "wb") as f:
    pickle.dump(window_mut_matrices, f)
print(f"\nSaved window_mut_matrices to: {save_path}")


Running SigProfilerMatrixGenerator for 120 combinations using 30 workers...
Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DI

## Reload: Skip Steps 1-3 (after kernel restart)

In [5]:
# ─────────────────────────────────────────────────────────────
# RELOAD: Skip Steps 1-3 if matrices already computed
# Run this instead of Steps 1-3 after a kernel restart
# ─────────────────────────────────────────────────────────────

save_path = os.path.join(OUT_SHARED, "window_mut_matrices.pkl")

if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        window_mut_matrices = pickle.load(f)
    print(f"Loaded window_mut_matrices from: {save_path}")
    for name in REGION_FILES:
        print(f"  {name}: {len(window_mut_matrices.get(name, {}))} windows")
else:
    # Fallback: reload from SBS96 .all files on disk
    print("Pickle not found — reloading from .all files...")
    window_mut_matrices = {name: {} for name in REGION_FILES}
    for name in REGION_FILES:
        for w in range(N_WINDOWS):
            vcf_dir = os.path.join(TMP_NORM, f"{name}_w{w:02d}")
            if not os.path.isdir(vcf_dir):
                continue
            for root, dirs, files in os.walk(vcf_dir):
                for fname in files:
                    if "SBS96" in fname and fname.endswith(".all"):
                        try:
                            mat = pd.read_csv(os.path.join(root, fname), sep="\t", index_col=0)
                            window_mut_matrices[name][w] = mat.sum(axis=1)
                        except Exception as e:
                            print(f"  Warning: {fname}: {e}")
    for name in REGION_FILES:
        print(f"  {name}: {len(window_mut_matrices[name])} windows loaded")


Loaded window_mut_matrices from: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_nothresh/window_mut_matrices.pkl
  All_protein_coding_genes: 20 windows
  lncRNA_expressed_in_testes: 20 windows
  Protein_coding_genes_not_expressed_in_testes: 20 windows
  lncRNA_not_expressed_in_testes: 20 windows
  Intergenic_RNAPII_pause_sites: 20 windows
  Random_intergenic: 20 windows


## Step 4 (NORM): Compute Trinucleotide Opportunities
*(Skip and use Reload cell below if already computed)*

In [6]:
# ─────────────────────────────────────────────────────────────
# STEP 4 (NORM): Compute trinucleotide opportunities per window via BED files
# ─────────────────────────────────────────────────────────────

def load_bed(bed_path):
    regions = []
    STANDARD = set([f"chr{i}" for i in range(1,23)] + ["chrX","chrY","chrM"])
    with open(bed_path) as f:
        for line in f:
            cols = line.strip().split()
            if len(cols) < 3:
                continue
            chrom = cols[0]
            if chrom not in STANDARD:
                continue
            strand = cols[5] if len(cols) > 5 else "+"
            regions.append((chrom, int(cols[1]), int(cols[2]), strand))
    return regions

def compute_window_bed_files(region_name, bed_path):
    regions = load_bed(bed_path)
    window_bed_dir = os.path.join(TMP_NORM, f"beds_{region_name}")
    os.makedirs(window_bed_dir, exist_ok=True)
    for w in range(N_WINDOWS):
        w_start, w_end = w * WINDOW_SIZE, (w + 1) * WINDOW_SIZE
        sub_bed = os.path.join(window_bed_dir, f"window_{w:02d}.bed")
        with open(sub_bed, "w") as f:
            for chrom, start, end, strand in regions:
                if strand == "+":
                    g_start, g_end = start + w_start, start + w_end
                else:
                    g_start, g_end = end - w_end, end - w_start
                g_start = max(0, g_start)
                g_end   = min(end, g_end)
                if g_end > g_start:
                    f.write(f"{chrom}\t{g_start}\t{g_end}\t.\t.\t{strand}\n")
    return window_bed_dir

def get_opp_for_window(args):
    region_name, w, vcf_dir, sub_bed_path = args
    try:
        matrices = matGen.SigProfilerMatrixGeneratorFunc(
            f"opp_{region_name}_w{w:02d}", "GRCh38", vcf_dir,
            exome=False, bed_file=sub_bed_path, chrom_based=False,
            plot=False, seqInfo=False,
        )
        if "96" in matrices:
            return (region_name, w, matrices["96"].sum(axis=1))
        return (region_name, w, None)
    except Exception:
        return (region_name, w, None)

print("Computing per-window trinucleotide opportunities...")
opp_tasks = []
for name in REGION_FILES:
    bed_path = os.path.join(BED_1KB_DIR, name + ".bed")
    if not os.path.exists(bed_path):
        print(f"  WARNING: No BED file for {name}")
        continue
    window_bed_dir = compute_window_bed_files(name, bed_path)
    for w in range(N_WINDOWS):
        sub_bed = os.path.join(window_bed_dir, f"window_{w:02d}.bed")
        vcf_dir = window_vcf_dirs[name].get(w, list(window_vcf_dirs[name].values())[0])
        opp_tasks.append((name, w, vcf_dir, sub_bed))

print(f"Running {len(opp_tasks)} opportunity calculations with {N_WORKERS} workers...")
all_window_opp_raw = {name: {} for name in REGION_FILES}

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(get_opp_for_window, t): t for t in opp_tasks}
    completed = 0
    for future in as_completed(futures):
        name, w, result = future.result()
        if result is not None:
            all_window_opp_raw[name][w] = result
        completed += 1
        if completed % 40 == 0:
            print(f"  {completed}/{len(opp_tasks)} complete...")

print("\nOpportunity computation complete.")
for name in REGION_FILES:
    print(f"  {name}: {len(all_window_opp_raw[name])} windows with opportunity data")

# ── SAVE opportunities ──
opp_save = os.path.join(OUT_SHARED, "window_opp_raw.pkl")
with open(opp_save, "wb") as f:
    pickle.dump(all_window_opp_raw, f)
print(f"Saved opportunities to: {opp_save}")


Computing per-window trinucleotide opportunities...
Running 120 opportunity calculations with 30 workers...
Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting matrix generation for SNVs and DINUCs...Starting m

## Reload: Skip Step 4 NORM (after kernel restart)

In [7]:
# ─────────────────────────────────────────────────────────────
# RELOAD: Skip Step 4 (NORM) if opportunities already computed
# ─────────────────────────────────────────────────────────────

opp_save = os.path.join(OUT_SHARED, "window_opp_raw.pkl")
if os.path.exists(opp_save):
    with open(opp_save, "rb") as f:
        all_window_opp_raw = pickle.load(f)
    print(f"Loaded opportunities from: {opp_save}")
    for name in REGION_FILES:
        print(f"  {name}: {len(all_window_opp_raw.get(name, {}))} windows")
else:
    print("Opportunity pickle not found — run Step 4 (NORM) first.")


Loaded opportunities from: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_nothresh/window_opp_raw.pkl
  All_protein_coding_genes: 20 windows
  lncRNA_expressed_in_testes: 20 windows
  Protein_coding_genes_not_expressed_in_testes: 20 windows
  lncRNA_not_expressed_in_testes: 20 windows
  Intergenic_RNAPII_pause_sites: 20 windows
  Random_intergenic: 20 windows


## Step 5 (NORM): Opportunity-Normalized Fitting
*(Skip and use Reload cell below if already computed)*

In [8]:
# ─────────────────────────────────────────────────────────────
# STEP 5 (NORM): Opportunity-normalized NNLS fitting
# ─────────────────────────────────────────────────────────────

cosmic       = pd.read_csv(COSMIC_REF, sep="\t", index_col=0)
sigs_to_fit  = cosmic.columns.tolist()
channels     = cosmic.index.tolist()
cosmic_mat   = cosmic.loc[channels].values

def fit_window_norm(observed_series, opp_series, cosmic_mat, sigs, threshold=0.05):
    observed = observed_series.reindex(channels, fill_value=0).values.astype(float)
    opp      = opp_series.reindex(channels, fill_value=0).values.astype(float)
    if opp.sum() > 0:
        opp_norm      = opp / opp.sum()
        observed_norm = observed / (opp_norm + 1e-10)
        cosmic_norm   = cosmic_mat / (opp_norm[:, np.newaxis] + 1e-10)
    else:
        observed_norm = observed
        cosmic_norm   = cosmic_mat
    acts = pd.Series(nnls(cosmic_norm, observed_norm)[0], index=sigs)
    total = acts.sum()
    if total > 0:
        acts[acts / total < threshold] = 0
    return acts

print("Fitting (opportunity-normalized)...")
all_window_activities_norm = {}

for name in REGION_FILES:
    print(f"  {name}")
    window_acts = {}
    for w in range(N_WINDOWS):
        if w not in window_mut_matrices[name]:
            window_acts[w] = pd.Series(0.0, index=sigs_to_fit)
            continue
        observed = window_mut_matrices[name][w].reindex(channels, fill_value=0)
        opp = all_window_opp_raw[name][w].reindex(channels, fill_value=0) \
              if all_window_opp_raw.get(name) and w in all_window_opp_raw[name] \
              else pd.Series(1.0, index=channels)
        window_acts[w] = fit_window_norm(observed, opp, cosmic_mat, sigs_to_fit, ACTIVITY_THRESHOLD)
    df_acts = pd.DataFrame(window_acts).T
    df_acts.index.name = "window"
    all_window_activities_norm[name] = df_acts
    active = df_acts.columns[(df_acts > 0).any(axis=0)].tolist()
    print(f"    Active: {active}")

# ── SAVE ──
save_path = os.path.join(OUT_NORM, "all_window_activities_norm.pkl")
with open(save_path, "wb") as f:
    pickle.dump(all_window_activities_norm, f)
print(f"\nSaved normalized activities to: {save_path}")


Fitting (opportunity-normalized)...
  All_protein_coding_genes
    Active: ['SBS1', 'SBS2', 'SBS3', 'SBS5', 'SBS6', 'SBS7a', 'SBS7b', 'SBS10a', 'SBS10b', 'SBS11', 'SBS12', 'SBS14', 'SBS17a', 'SBS17b', 'SBS19', 'SBS20', 'SBS21', 'SBS23', 'SBS24', 'SBS26', 'SBS28', 'SBS30', 'SBS32', 'SBS35', 'SBS39', 'SBS40b', 'SBS42', 'SBS43', 'SBS46', 'SBS48', 'SBS49', 'SBS53', 'SBS54', 'SBS55', 'SBS59', 'SBS60', 'SBS84', 'SBS86', 'SBS88', 'SBS96', 'SBS97', 'SBS98', 'SBS101', 'SBS106']
  lncRNA_expressed_in_testes
    Active: ['SBS1', 'SBS2', 'SBS3', 'SBS5', 'SBS6', 'SBS7a', 'SBS7b', 'SBS7d', 'SBS9', 'SBS10a', 'SBS10b', 'SBS11', 'SBS12', 'SBS13', 'SBS14', 'SBS16', 'SBS17a', 'SBS17b', 'SBS19', 'SBS20', 'SBS21', 'SBS23', 'SBS24', 'SBS26', 'SBS28', 'SBS30', 'SBS32', 'SBS35', 'SBS37', 'SBS39', 'SBS40b', 'SBS42', 'SBS43', 'SBS45', 'SBS46', 'SBS48', 'SBS49', 'SBS50', 'SBS53', 'SBS54', 'SBS55', 'SBS58', 'SBS59', 'SBS60', 'SBS84', 'SBS86', 'SBS87', 'SBS88', 'SBS91', 'SBS96', 'SBS97', 'SBS98', 'SBS101', 'SBS105

## Reload: Skip Step 5 NORM (after kernel restart)

In [9]:
# ─────────────────────────────────────────────────────────────
# RELOAD: Skip Step 5 (NORM) if activities already computed
# ─────────────────────────────────────────────────────────────

save_path = os.path.join(OUT_NORM, "all_window_activities_norm.pkl")
if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        all_window_activities_norm = pickle.load(f)
    print(f"Loaded normalized activities from: {save_path}")
else:
    print("Pickle not found — run Step 5 (NORM) first.")


Loaded normalized activities from: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_nothresh/all_window_activities_norm.pkl


## Step 5 (NO-NORM): Fitting Without Opportunity Normalization
*(Skip and use Reload cell below if already computed)*

In [10]:
# ─────────────────────────────────────────────────────────────
# STEP 5 (NO-NORM): NNLS fitting without opportunity normalization
# ─────────────────────────────────────────────────────────────

cosmic       = pd.read_csv(COSMIC_REF, sep="\t", index_col=0)
sigs_to_fit  = cosmic.columns.tolist()
channels     = cosmic.index.tolist()
cosmic_mat   = cosmic.loc[channels].values

def fit_window_no_norm(observed_series, cosmic_mat, sigs, threshold=0.05):
    observed = observed_series.reindex(channels, fill_value=0).values.astype(float)
    acts     = pd.Series(nnls(cosmic_mat, observed)[0], index=sigs)
    total    = acts.sum()
    if total > 0:
        acts[acts / total < threshold] = 0
    return acts

print("Fitting (no opportunity normalization)...")
all_window_activities_nonorm = {}

for name in REGION_FILES:
    print(f"  {name}")
    window_acts = {}
    for w in range(N_WINDOWS):
        if w not in window_mut_matrices[name]:
            window_acts[w] = pd.Series(0.0, index=sigs_to_fit)
            continue
        observed = window_mut_matrices[name][w].reindex(channels, fill_value=0)
        window_acts[w] = fit_window_no_norm(observed, cosmic_mat, sigs_to_fit, ACTIVITY_THRESHOLD)
    df_acts = pd.DataFrame(window_acts).T
    df_acts.index.name = "window"
    all_window_activities_nonorm[name] = df_acts
    active = df_acts.columns[(df_acts > 0).any(axis=0)].tolist()
    print(f"    Active: {active}")

# ── SAVE ──
save_path = os.path.join(OUT_NONORM, "all_window_activities_nonorm.pkl")
with open(save_path, "wb") as f:
    pickle.dump(all_window_activities_nonorm, f)
print(f"\nSaved no-norm activities to: {save_path}")


Fitting (no opportunity normalization)...
  All_protein_coding_genes
    Active: ['SBS2', 'SBS3', 'SBS5', 'SBS6', 'SBS7b', 'SBS10a', 'SBS11', 'SBS12', 'SBS15', 'SBS17a', 'SBS17b', 'SBS21', 'SBS22a', 'SBS23', 'SBS24', 'SBS26', 'SBS30', 'SBS31', 'SBS32', 'SBS35', 'SBS39', 'SBS43', 'SBS46', 'SBS48', 'SBS49', 'SBS54', 'SBS55', 'SBS84', 'SBS86', 'SBS87', 'SBS88', 'SBS94', 'SBS97', 'SBS98', 'SBS101', 'SBS102', 'SBS106']
  lncRNA_expressed_in_testes
    Active: ['SBS1', 'SBS2', 'SBS3', 'SBS5', 'SBS6', 'SBS7a', 'SBS7b', 'SBS7d', 'SBS9', 'SBS10a', 'SBS10b', 'SBS11', 'SBS12', 'SBS13', 'SBS15', 'SBS16', 'SBS17a', 'SBS17b', 'SBS19', 'SBS21', 'SBS23', 'SBS24', 'SBS26', 'SBS28', 'SBS30', 'SBS31', 'SBS32', 'SBS33', 'SBS35', 'SBS39', 'SBS43', 'SBS46', 'SBS48', 'SBS49', 'SBS54', 'SBS55', 'SBS57', 'SBS60', 'SBS84', 'SBS86', 'SBS87', 'SBS88', 'SBS91', 'SBS94', 'SBS97', 'SBS98', 'SBS101', 'SBS102', 'SBS106']
  Protein_coding_genes_not_expressed_in_testes
    Active: ['SBS1', 'SBS2', 'SBS3', 'SBS4', 'SBS5'

## Reload: Skip Step 5 NO-NORM (after kernel restart)

In [11]:
# ─────────────────────────────────────────────────────────────
# RELOAD: Skip Step 5 (NO-NORM) if activities already computed
# ─────────────────────────────────────────────────────────────

save_path = os.path.join(OUT_NONORM, "all_window_activities_nonorm.pkl")
if os.path.exists(save_path):
    with open(save_path, "rb") as f:
        all_window_activities_nonorm = pickle.load(f)
    print(f"Loaded no-norm activities from: {save_path}")
else:
    print("Pickle not found — run Step 5 (NO-NORM) first.")


Loaded no-norm activities from: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_no_norm_nothresh/all_window_activities_nonorm.pkl


## Colors: Build Shared Color Map
*(Run once per session)*

In [19]:
# ─────────────────────────────────────────────────────────────
# COLORS: Build shared color map from UNIVERSAL_ORDER
# Run once; rerun only if you want to change colors
# ─────────────────────────────────────────────────────────────

def to_pastel(color, saturation=0.3, lightness_boost=0.4):
    r, g, b, a = color
    h, s, v = mcolors.rgb_to_hsv([r, g, b])
    return (*mcolors.hsv_to_rgb([h, s * saturation, min(1.0, v + lightness_boost)]), a)

# Use 0.05 to 0.95 range to avoid black and white at the extremes
n_total         = len(UNIVERSAL_ORDER)
color_positions = np.linspace(0.05, 0.95, n_total)
base_cmap       = plt.cm.get_cmap("nipy_spectral")
sig_colors      = {sig: to_pastel(base_cmap(color_positions[i]))
                   for i, sig in enumerate(UNIVERSAL_ORDER)}

print("sig_colors built for", len(sig_colors), "signatures")
for sig, c in sig_colors.items():
    print(f"  {sig}: {tuple(round(x,3) for x in c)}")

sig_colors built for 83 signatures
  SBS48: (np.float64(0.868), np.float64(0.631), np.float64(0.902), np.float64(1.0))
  SBS91: (np.float64(0.91), np.float64(0.662), np.float64(0.945), np.float64(1.0))
  SBS59: (np.float64(0.927), np.float64(0.673), np.float64(0.961), np.float64(1.0))
  SBS106: (np.float64(0.943), np.float64(0.684), np.float64(0.976), np.float64(1.0))
  SBS43: (np.float64(0.959), np.float64(0.695), np.float64(0.992), np.float64(1.0))
  SBS55: (np.float64(0.955), np.float64(0.7), np.float64(1.0), np.float64(1.0))
  SBS60: (np.float64(0.888), np.float64(0.7), np.float64(1.0), np.float64(1.0))
  SBS10b: (np.float64(0.824), np.float64(0.7), np.float64(1.0), np.float64(1.0))
  SBS2: (np.float64(0.763), np.float64(0.7), np.float64(1.0), np.float64(1.0))
  SBS17b: (np.float64(0.705), np.float64(0.7), np.float64(1.0), np.float64(1.0))
  SBS28: (np.float64(0.7), np.float64(0.7), np.float64(1.0), np.float64(1.0))
  SBS57: (np.float64(0.7), np.float64(0.7), np.float64(1.0), np.fl

## Plot: Opportunity-Normalized Version
*(Edit and rerun freely)*

In [20]:
# ─────────────────────────────────────────────────────────────
# PLOT: Opportunity-Normalized — edit and rerun freely
# ─────────────────────────────────────────────────────────────

all_window_activities = all_window_activities_norm  # point to correct version

# Build active sigs for this version
all_active_sigs = sorted(set(
    s for df in all_window_activities.values()
    for s in df.columns[(df > 0).any(axis=0)]
))
clustered_sigs = [s for s in UNIVERSAL_ORDER if s in all_active_sigs]
print(f"Active signatures ({len(clustered_sigs)}): {clustered_sigs}")

ncols, nrows = 3, 2
fig, axes = plt.subplots(nrows, ncols, figsize=(33, nrows * 5), sharey=False)
axes = axes.flatten()

MIN_PCT_TO_LABEL = 8.0

for idx, name in enumerate(PLOT_ORDER):
    ax = axes[idx]
    df = all_window_activities[name]

    df_pct = df.div(df.sum(axis=1).replace(0, np.nan), axis=0) * 100
    df_pct = df_pct.fillna(0)

    active_here = [s for s in clustered_sigs if s in df_pct.columns and df_pct[s].sum() > 0]
    bottom = np.zeros(N_WINDOWS)
    bar_segments = {}

    for sig in active_here:
        vals = df_pct[sig].reindex(range(N_WINDOWS), fill_value=0).values
        ax.bar(range(N_WINDOWS), vals, bottom=bottom,
               color=sig_colors[sig], width=0.85, edgecolor="none")
        bar_segments[sig] = (bottom.copy(), vals.copy())
        bottom += vals

    for sig, (bot, heights) in bar_segments.items():
        for w in range(N_WINDOWS):
            h = heights[w]
            if h >= MIN_PCT_TO_LABEL:
                ax.text(w, bot[w] + h / 2, sig,
                        ha="center", va="center", fontsize=8,
                        color="black", rotation=0, clip_on=True)

    ax.set_title(name.replace("_", " "), fontsize=18, fontweight="bold")
    ax.set_xlabel("Position relative to TSS/focal site", fontsize=18)
    ax.set_ylabel("% Mutations", fontsize=18)
    ax.tick_params(axis="y", labelsize=14)
    ax.set_xticks([w + 0.5 for w in range(N_WINDOWS)])
    xtick_labels = [f"{((w+1)*50) - 500:+d}" for w in range(N_WINDOWS)]
    xtick_labels = [l.replace("+0", "0") for l in xtick_labels]
    ax.set_xticklabels(xtick_labels, rotation=90, fontsize=14)
    ax.set_ylim(0, 100)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for idx in range(len(PLOT_ORDER), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("SBS Signature Activities Across 50bp Windows\n(Opportunity-Normalized)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
out = os.path.join(OUT_NORM, "sbs_windowed_stacked_norm.svg")
plt.savefig(out, format="svg", bbox_inches="tight")
print(f"Saved: {out}")
plt.show()

# Legend
ncols_leg = 5
fig_leg, ax_leg = plt.subplots(figsize=(ncols_leg * 3, int(np.ceil(len(clustered_sigs)/ncols_leg)) * 0.6 + 0.5))
ax_leg.set_axis_off()
ax_leg.legend(
    handles=[mpatches.Patch(color=sig_colors[s], label=s) for s in clustered_sigs],
    title="SBS Signature (ordered by similarity)",
    loc="center", ncol=ncols_leg, fontsize=10, title_fontsize=11,
    frameon=True, handlelength=2, handleheight=1.2,
)
plt.tight_layout()
leg_out = os.path.join(OUT_NORM, "sbs_windowed_stacked_norm_legend.svg")
plt.savefig(leg_out, format="svg", bbox_inches="tight")
print(f"Legend saved: {leg_out}")
plt.show()


Active signatures (79): ['SBS48', 'SBS91', 'SBS59', 'SBS106', 'SBS43', 'SBS55', 'SBS60', 'SBS10b', 'SBS2', 'SBS17b', 'SBS28', 'SBS57', 'SBS17a', 'SBS33', 'SBS54', 'SBS27', 'SBS98', 'SBS1', 'SBS87', 'SBS13', 'SBS103', 'SBS39', 'SBS86', 'SBS49', 'SBS53', 'SBS10c', 'SBS10a', 'SBS10d', 'SBS14', 'SBS20', 'SBS35', 'SBS24', 'SBS29', 'SBS45', 'SBS8', 'SBS94', 'SBS100', 'SBS4', 'SBS107', 'SBS6', 'SBS99', 'SBS42', 'SBS84', 'SBS105', 'SBS97', 'SBS31', 'SBS19', 'SBS23', 'SBS7a', 'SBS7b', 'SBS102', 'SBS30', 'SBS110', 'SBS40a', 'SBS11', 'SBS32', 'SBS58', 'SBS50', 'SBS51', 'SBS34', 'SBS85', 'SBS7d', 'SBS21', 'SBS46', 'SBS37', 'SBS12', 'SBS26', 'SBS16', 'SBS88', 'SBS101', 'SBS25', 'SBS40b', 'SBS9', 'SBS96', 'SBS92', 'SBS40c', 'SBS5', 'SBS3', 'SBS89']
Saved: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_nothresh/sbs_windowed_stacked_norm.svg
Legend saved: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_nothresh/sbs_windowed_stacked_norm_legend.svg


## Find all SBS (universal)

In [17]:
# Recompute UNIVERSAL_ORDER from current activities (no threshold)
sigs_norm   = sorted(set(
    s for df in all_window_activities_norm.values()
    for s in df.columns[(df > 0).any(axis=0)]
))

sigs_nonorm = sorted(set(
    s for df in all_window_activities_nonorm.values()
    for s in df.columns[(df > 0).any(axis=0)]
))

all_sigs_union = sorted(set(sigs_norm) | set(sigs_nonorm))
print(f"Norm active ({len(sigs_norm)}): {sigs_norm}")
print(f"No-norm active ({len(sigs_nonorm)}): {sigs_nonorm}")
print(f"Union ({len(all_sigs_union)}): {all_sigs_union}")

# Cluster on union
cosmic = pd.read_csv(COSMIC_REF, sep="\t", index_col=0)
cosmic_union = cosmic[[s for s in all_sigs_union if s in cosmic.columns]]

n = len(all_sigs_union)
sim = pd.DataFrame(np.zeros((n, n)), index=all_sigs_union, columns=all_sigs_union)
for s1 in all_sigs_union:
    for s2 in all_sigs_union:
        sim.loc[s1, s2] = 1 - cosine(cosmic_union[s1].values, cosmic_union[s2].values)

dist  = squareform(1 - sim.values, checks=False)
order = leaves_list(linkage(dist, method="average"))
NEW_UNIVERSAL_ORDER = [all_sigs_union[i] for i in order]

print(f"\nNew UNIVERSAL_ORDER ({len(NEW_UNIVERSAL_ORDER)}):")
print(NEW_UNIVERSAL_ORDER)

Norm active (79): ['SBS1', 'SBS100', 'SBS101', 'SBS102', 'SBS103', 'SBS105', 'SBS106', 'SBS107', 'SBS10a', 'SBS10b', 'SBS10c', 'SBS10d', 'SBS11', 'SBS110', 'SBS12', 'SBS13', 'SBS14', 'SBS16', 'SBS17a', 'SBS17b', 'SBS19', 'SBS2', 'SBS20', 'SBS21', 'SBS23', 'SBS24', 'SBS25', 'SBS26', 'SBS27', 'SBS28', 'SBS29', 'SBS3', 'SBS30', 'SBS31', 'SBS32', 'SBS33', 'SBS34', 'SBS35', 'SBS37', 'SBS39', 'SBS4', 'SBS40a', 'SBS40b', 'SBS40c', 'SBS42', 'SBS43', 'SBS45', 'SBS46', 'SBS48', 'SBS49', 'SBS5', 'SBS50', 'SBS51', 'SBS53', 'SBS54', 'SBS55', 'SBS57', 'SBS58', 'SBS59', 'SBS6', 'SBS60', 'SBS7a', 'SBS7b', 'SBS7d', 'SBS8', 'SBS84', 'SBS85', 'SBS86', 'SBS87', 'SBS88', 'SBS89', 'SBS9', 'SBS91', 'SBS92', 'SBS94', 'SBS96', 'SBS97', 'SBS98', 'SBS99']
No-norm active (77): ['SBS1', 'SBS101', 'SBS102', 'SBS103', 'SBS105', 'SBS106', 'SBS107', 'SBS10a', 'SBS10b', 'SBS10c', 'SBS10d', 'SBS11', 'SBS110', 'SBS12', 'SBS13', 'SBS14', 'SBS15', 'SBS16', 'SBS17a', 'SBS17b', 'SBS19', 'SBS2', 'SBS20', 'SBS21', 'SBS22a', 'S

## Plot: No-Normalization Version
*(Edit and rerun freely)*

In [21]:
# ─────────────────────────────────────────────────────────────
# PLOT: No Opportunity Normalization — edit and rerun freely
# ─────────────────────────────────────────────────────────────

all_window_activities = all_window_activities_nonorm  # point to correct version

# Build active sigs for this version
all_active_sigs = sorted(set(
    s for df in all_window_activities.values()
    for s in df.columns[(df > 0).any(axis=0)]
))
clustered_sigs = [s for s in UNIVERSAL_ORDER if s in all_active_sigs]
print(f"Active signatures ({len(clustered_sigs)}): {clustered_sigs}")

ncols, nrows = 3, 2
fig, axes = plt.subplots(nrows, ncols, figsize=(33, nrows * 5), sharey=False)
axes = axes.flatten()

MIN_PCT_TO_LABEL = 8.0

for idx, name in enumerate(PLOT_ORDER):
    ax = axes[idx]
    df = all_window_activities[name]

    df_pct = df.div(df.sum(axis=1).replace(0, np.nan), axis=0) * 100
    df_pct = df_pct.fillna(0)

    active_here = [s for s in clustered_sigs if s in df_pct.columns and df_pct[s].sum() > 0]
    bottom = np.zeros(N_WINDOWS)
    bar_segments = {}

    for sig in active_here:
        vals = df_pct[sig].reindex(range(N_WINDOWS), fill_value=0).values
        ax.bar(range(N_WINDOWS), vals, bottom=bottom,
               color=sig_colors[sig], width=0.85, edgecolor="none")
        bar_segments[sig] = (bottom.copy(), vals.copy())
        bottom += vals

    for sig, (bot, heights) in bar_segments.items():
        for w in range(N_WINDOWS):
            h = heights[w]
            if h >= MIN_PCT_TO_LABEL:
                ax.text(w, bot[w] + h / 2, sig,
                        ha="center", va="center", fontsize=8,
                        color="black", rotation=0, clip_on=True)

    ax.set_title(name.replace("_", " "), fontsize=18, fontweight="bold")
    ax.set_xlabel("Position relative to TSS/focal site", fontsize=18)
    ax.set_ylabel("% Mutations", fontsize=18)
    ax.tick_params(axis="y", labelsize=14)
    ax.set_xticks([w + 0.5 for w in range(N_WINDOWS)])
    xtick_labels = [f"{((w+1)*50) - 500:+d}" for w in range(N_WINDOWS)]
    xtick_labels = [l.replace("+0", "0") for l in xtick_labels]
    ax.set_xticklabels(xtick_labels, rotation=90, fontsize=14)
    ax.set_ylim(0, 100)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for idx in range(len(PLOT_ORDER), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("SBS Signature Activities Across 50bp Windows\n(No Opportunity Normalization)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
out = os.path.join(OUT_NONORM, "sbs_windowed_stacked_no_norm.svg")
plt.savefig(out, format="svg", bbox_inches="tight")
print(f"Saved: {out}")
plt.show()

# Legend
ncols_leg = 5
fig_leg, ax_leg = plt.subplots(figsize=(ncols_leg * 3, int(np.ceil(len(clustered_sigs)/ncols_leg)) * 0.6 + 0.5))
ax_leg.set_axis_off()
ax_leg.legend(
    handles=[mpatches.Patch(color=sig_colors[s], label=s) for s in clustered_sigs],
    title="SBS Signature (ordered by similarity)",
    loc="center", ncol=ncols_leg, fontsize=10, title_fontsize=11,
    frameon=True, handlelength=2, handleheight=1.2,
)
plt.tight_layout()
leg_out = os.path.join(OUT_NONORM, "sbs_windowed_stacked_no_norm_legend.svg")
plt.savefig(leg_out, format="svg", bbox_inches="tight")
print(f"Legend saved: {leg_out}")
plt.show()


Active signatures (77): ['SBS48', 'SBS91', 'SBS59', 'SBS106', 'SBS43', 'SBS55', 'SBS60', 'SBS10b', 'SBS2', 'SBS17b', 'SBS28', 'SBS57', 'SBS17a', 'SBS33', 'SBS54', 'SBS98', 'SBS1', 'SBS87', 'SBS13', 'SBS103', 'SBS39', 'SBS86', 'SBS49', 'SBS53', 'SBS10c', 'SBS10a', 'SBS10d', 'SBS14', 'SBS20', 'SBS35', 'SBS24', 'SBS29', 'SBS8', 'SBS94', 'SBS4', 'SBS107', 'SBS15', 'SBS6', 'SBS99', 'SBS42', 'SBS84', 'SBS105', 'SBS44', 'SBS97', 'SBS31', 'SBS19', 'SBS23', 'SBS7a', 'SBS7b', 'SBS102', 'SBS30', 'SBS110', 'SBS11', 'SBS32', 'SBS58', 'SBS50', 'SBS51', 'SBS7c', 'SBS34', 'SBS85', 'SBS7d', 'SBS21', 'SBS46', 'SBS37', 'SBS12', 'SBS26', 'SBS16', 'SBS88', 'SBS22a', 'SBS101', 'SBS9', 'SBS96', 'SBS92', 'SBS40c', 'SBS5', 'SBS3', 'SBS89']
Saved: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_no_norm_nothresh/sbs_windowed_stacked_no_norm.svg
Legend saved: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_no_somatic_dnm_no_norm_nothresh/sbs_windowed_stacked_no_norm_legend.svg


## Shared Legend (all union signatures)

In [22]:
# ─────────────────────────────────────────────────────────────
# SHARED LEGEND: all signatures from union of both versions
# ─────────────────────────────────────────────────────────────

ncols_leg = 5
fig_leg, ax_leg = plt.subplots(figsize=(ncols_leg * 3, int(np.ceil(len(UNIVERSAL_ORDER)/ncols_leg)) * 0.6 + 0.5))
ax_leg.set_axis_off()
ax_leg.legend(
    handles=[mpatches.Patch(color=sig_colors[s], label=s)
             for s in UNIVERSAL_ORDER if s in sig_colors],
    title="SBS Signature (ordered by similarity across both versions)",
    loc="center", ncol=ncols_leg, fontsize=10, title_fontsize=11,
    frameon=True, handlelength=2, handleheight=1.2,
)
plt.tight_layout()
leg_out = os.path.join(OUT_SHARED, "sbs_shared_legend.svg")
plt.savefig(leg_out, format="svg", bbox_inches="tight")
print(f"Shared legend saved: {leg_out}")
plt.show()


Shared legend saved: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_nothresh/sbs_shared_legend.svg


## Pairwise Cosine Similarity Heatmap

In [23]:
# ─────────────────────────────────────────────────────────────
# PAIRWISE COSINE SIMILARITY: all union signatures
# ─────────────────────────────────────────────────────────────

cosmic       = pd.read_csv(COSMIC_REF, sep="\t", index_col=0)
cosmic_union = cosmic[[s for s in UNIVERSAL_ORDER if s in cosmic.columns]]

n = len(UNIVERSAL_ORDER)
pairwise_sim = pd.DataFrame(np.zeros((n, n)), index=UNIVERSAL_ORDER, columns=UNIVERSAL_ORDER)
for s1 in UNIVERSAL_ORDER:
    for s2 in UNIVERSAL_ORDER:
        pairwise_sim.loc[s1, s2] = 1 - cosine(cosmic_union[s1].values, cosmic_union[s2].values)

pairwise_sim = pairwise_sim.astype(float)

fig, ax = plt.subplots(figsize=(max(8, n * 0.65), max(7, n * 0.65)))
sns.heatmap(pairwise_sim, ax=ax, cmap="GnBu", vmin=0, vmax=1,
            linewidths=0.5, linecolor="lightgrey", annot=True, fmt=".2f",
            square=True, cbar_kws={"label": "Cosine Similarity", "shrink": 0.6},
            xticklabels=False, yticklabels=False)
ax.xaxis.set_ticks_position('bottom')
ax.xaxis.set_label_position('bottom')
ax.set_title("Pairwise Cosine Similarity Between COSMIC SBS Reference Profiles",
             fontsize=13, fontweight="bold", pad=40)

for i, sig in enumerate(UNIVERSAL_ORDER):
    color = sig_colors.get(sig, (0.9, 0.9, 0.9, 1.0))

    # X axis — below the heatmap (increase negative value to push further down)
    ax.text(
        i + 0.5, n + 0.3, sig,
        ha="center", va="top",
        fontsize=14, fontweight="bold",
        rotation=90,
        bbox=dict(boxstyle="square,pad=0.6", facecolor=color, edgecolor="none"),
    )

    # Y axis — left of heatmap (unchanged)
    ax.text(
        -0.3, i + 0.5, sig,
        ha="right", va="center",
        fontsize=14, fontweight="bold",
        bbox=dict(boxstyle="square,pad=0.6", facecolor=color, edgecolor="none"),
    )

ax.tick_params(axis="both", length=0)
plt.tight_layout()
out = os.path.join(OUT_SHARED, "sbs_union_pairwise_cosine.svg")
plt.savefig(out, format="svg", bbox_inches="tight")
print(f"Saved: {out}")
plt.show()
pairwise_sim.to_csv(out.replace(".svg", ".csv"))


Saved: /home/alexpalazzo1/Documents/Tina/COSMIC/output_windowed_nothresh/sbs_union_pairwise_cosine.svg
